<a href="https://colab.research.google.com/github/DevKaiqueTeixeira/IA/blob/main/SummitCodigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""UMC Summit: regressão linear e SVM/SVR com validação temporal.

Uso: python analise_literary_digest.py LitDigestFull.xlsx resultados
Bibliotecas: pandas, numpy, scikit-learn, matplotlib, openpyxl.
"""
from pathlib import Path
import json
import sys
import hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CONFIG = {
    1924: {'inicio':1, 'pesquisa':[0,2,3,10], 'real':[21,23,26,43,22]},
    1928: {'inicio':2, 'pesquisa':[0,2,10,23], 'real':[25,27,30,44,26]},
    1932: {'inicio':2, 'pesquisa':[0,2,9,24], 'real':[27,32,29,43,28]},
    1936: {'inicio':2, 'pesquisa':[0,2,10,32], 'real':[34,39,36,53,35]},
}
ALIASES = {'N. HAMPSHIRE':'NEW HAMPSHIRE','DIST COLUMBIA':'DISTRICT OF COLUMBIA'}

def normalizar(serie):
    return serie.astype(str).str.upper().str.strip().str.replace(r'\s+', ' ', regex=True).replace(ALIASES)

def extrair(arquivo, ano):
    cfg=CONFIG[ano]
    raw=pd.read_excel(arquivo,sheet_name=str(ano),header=None)
    a=raw.iloc[1:,cfg['real']].copy()
    a.columns=['estado','real_r','real_d','real_total','eleitorais']
    a.estado=normalizar(a.estado)
    for col in a.columns.drop('estado'):a[col]=pd.to_numeric(a[col],errors='coerce')
    a=a[a.real_r.notna() & a.real_d.notna() & ~a.estado.str.contains('TOTAL')].copy()
    if len(a)!=48 or a.estado.nunique()!=48:
        raise ValueError(f'{ano}: esperado 48 estados oficiais únicos')
    p=raw.iloc[cfg['inicio']:,cfg['pesquisa']].copy()
    p.columns=['estado','pesquisa_r','pesquisa_d','pesquisa_total']
    p['estado_original']=p.estado
    p['linha_excel']=p.index+1
    p.estado=normalizar(p.estado)
    for col in ['pesquisa_r','pesquisa_d','pesquisa_total']:p[col]=pd.to_numeric(p[col],errors='coerce')
    exclusoes=[]
    for _,r in p.loc[~p.estado.isin(a.estado)].iterrows():
        motivo=('Estado não informado' if 'UNKNOWN' in r.estado else
                'Distrito de Columbia sem resultado comparável' if 'COLUMBIA' in r.estado else
                'Totalizador; retirar para não duplicar respostas')
        exclusoes.append({'ano':ano,'linha_excel':int(r.linha_excel),'rotulo':r.estado,
                          'respostas':int(r.pesquisa_total) if pd.notna(r.pesquisa_total) else None,'motivo':motivo})
    limpo=p[p.estado.isin(a.estado)].merge(a,on='estado',validate='one_to_one').copy()
    contagens=['pesquisa_r','pesquisa_d','pesquisa_total','real_r','real_d','real_total','eleitorais']
    if len(limpo)!=48 or limpo[contagens].isna().any().any() or (limpo[contagens]<0).any().any():
        raise ValueError(f'{ano}: dados incompletos ou inválidos')
    for prefixo in ['pesquisa','real']:
        denom=limpo[f'{prefixo}_r']+limpo[f'{prefixo}_d']
        if (denom<=0).any() or (denom>limpo[f'{prefixo}_total']).any():raise ValueError('Totais incoerentes')
        limpo[f'{prefixo}_rd_pct']=100*limpo[f'{prefixo}_r']/denom
    limpo['ano']=ano
    limpo['nome_padronizado']=limpo.estado_original.astype(str).str.strip()!=limpo.estado
    meta={'ano':ano,'linhas_brutas':len(raw),'colunas_brutas':raw.shape[1],
          'estados':48,'respostas_estados':int(limpo.pesquisa_total.sum()),
          'votos_rd_oficiais':int((limpo.real_r+limpo.real_d).sum()),
          'outros_real_pct':float(100*(1-(limpo.real_r+limpo.real_d).sum()/limpo.real_total.sum())),
          'pesquisa_rd_pct':float(100*limpo.pesquisa_r.sum()/(limpo.pesquisa_r+limpo.pesquisa_d).sum()),
          'real_rd_pct':float(100*limpo.real_r.sum()/(limpo.real_r+limpo.real_d).sum())}
    return limpo,exclusoes,meta

def prever(modelo,X):
    return np.clip(modelo.predict(X),0,100)

def metricas(y,p):
    e=np.asarray(p)-np.asarray(y)
    return {'mae_pp':float(np.mean(abs(e))),'rmse_pp':float(np.sqrt(np.mean(e**2))),
            'acertos_estados':int(((np.asarray(p)>.5*100)==(np.asarray(y)>.5*100)).sum())}

def novo_svr(kernel,C):
    return make_pipeline(StandardScaler(),SVR(kernel=kernel,C=float(C),epsilon=1.0,gamma='scale'))


ESQUEMAS={
    'historico':['real_anterior'],
    'historico_e_pesquisa':['real_anterior','pesquisa_rd_pct'],
    'historico_e_variacao':['real_anterior','variacao_pesquisa'],
    'historico_e_duas_pesquisas':['real_anterior','pesquisa_rd_pct','pesquisa_anterior'],
}

def pares_temporais(base):
    anterior=base[['estado','ano','real_rd_pct','pesquisa_rd_pct','real_r','real_d']].copy()
    anterior.ano+=4
    anterior.columns=['estado','ano','real_anterior','pesquisa_anterior','r_anterior','d_anterior']
    pares=base.merge(anterior,on=['estado','ano'],validate='one_to_one')
    pares['variacao_pesquisa']=pares.pesquisa_rd_pct-pares.pesquisa_anterior
    return pares

def executar(arquivo,saida):
    saida=Path(saida);saida.mkdir(parents=True,exist_ok=True)
    bases=[];exclusoes=[];auditoria=[]
    for ano in [1924,1928,1932]:
        base,exc,meta=extrair(arquivo,ano);bases.append(base);exclusoes.extend(exc);auditoria.append(meta)
    historico=pd.concat(bases,ignore_index=True)
    pares=pares_temporais(historico)
    treino=pares[pares.ano==1928];validacao=pares[pares.ano==1932]
    busca=[]
    for esquema,cols in ESQUEMAS.items():
        for tipo in ['linear','svr']:
            configs=[(None,None)] if tipo=='linear' else [(k,c) for k in ['linear','rbf'] for c in [1,10,100]]
            for kernel,C in configs:
                mod=LinearRegression() if tipo=='linear' else novo_svr(kernel,C)
                mod.fit(treino[cols],treino.real_rd_pct)
                p=prever(mod,validacao[cols])
                busca.append({'tipo':tipo,'esquema':esquema,'n_features':len(cols),'kernel':kernel,'C':C,
                              'ano_alvo_treino':1928,'ano_validacao':1932,**metricas(validacao.real_rd_pct,p)})
    selecao=pd.DataFrame(busca)
    picks={}
    for tipo in ['linear','svr']:
        r=selecao[selecao.tipo==tipo].sort_values(['mae_pp','n_features','C'],na_position='first').iloc[0]
        picks[tipo]={'esquema':r.esquema,'features':ESQUEMAS[r.esquema],
                     'kernel':None if pd.isna(r.kernel) else str(r.kernel),
                     'C':None if pd.isna(r.C) else float(r.C),'epsilon_pp':1.0 if tipo=='svr' else None,
                     'mae_validacao_1932':float(r.mae_pp),'acertos_validacao_1932':int(r.acertos_estados)}
    protocolo={'anos_usados':[1924,1928,1932],'n_estados_por_ano':48,
        'treino_validacao':'1924 fornece atributos anteriores de 1928; ajustar em 1928 e avaliar em 1932 usando 1928 como referência.',
        'treino_final':'96 pares estado-eleição: 1924→1928 e 1928→1932.',
        'previsao':'1932→1936, usando o resultado de 1932 e pesquisas de 1932/1936.',
        'selecao':'Menor MAE em 1932; empate: menos atributos e menor C.',
        'linear':picks['linear'],'svr':picks['svr'],
        'limite':'Só uma eleição de validação. Estudo retrospectivo; 1936 já havia sido analisado, mas não define os parâmetros.'}
    (saida/'protocolo_antes_avaliacao_1936.json').write_text(json.dumps(protocolo,ensure_ascii=False,indent=2))
    selecao.to_csv(saida/'selecao_modelos_em_1932.csv',index=False,encoding='utf-8-sig')
    modelos={}
    for tipo,pick in picks.items():
        mod=LinearRegression() if tipo=='linear' else novo_svr(pick['kernel'],pick['C'])
        mod.fit(pares[pick['features']],pares.real_rd_pct)
        modelos[tipo]=mod
    # Nenhum voto oficial de 1936 é passado ao fit ou ao predict.
    b36,exc,meta=extrair(arquivo,1936);exclusoes.extend(exc);auditoria.append(meta)
    completa=pd.concat([historico,b36],ignore_index=True)
    teste=pares_temporais(completa);teste=teste[teste.ano==1936].copy()
    for tipo,mod in modelos.items():teste[tipo]=prever(mod,teste[picks[tipo]['features']])
    teste['peso_1932']=teste.r_anterior+teste.d_anterior
    resultados=[]
    for nome,col in [('Pesquisa','pesquisa_rd_pct'),('Referência 1932','real_anterior'),('Regressão linear','linear'),('SVM / SVR','svr')]:
        p=teste[col];teste[f'vencedor_{col}']=np.where(p>50,'Landon','Roosevelt');teste[f'erro_{col}_pp']=p-teste.real_rd_pct
        evr=int(teste.loc[p>50,'eleitorais'].sum());evd=int(teste.eleitorais.sum()-evr)
        nacional=float(np.average(p,weights=teste.peso_1932))
        resultados.append({'metodo':nome,'coluna':col,**metricas(teste.real_rd_pct,p),
                            'r_pct_pesos_1932':nacional,'d_pct_pesos_1932':100-nacional,
                            'vencedor_popular':'Landon' if nacional>50 else 'Roosevelt',
                            'eleitorais_landon':evr,'eleitorais_roosevelt':evd,
                            'vencedor_eleitoral':'Landon' if evr>evd else 'Roosevelt'})
    lr=modelos['linear']
    resumo={'protocolo':protocolo,'resultados_1936':resultados,'auditoria':auditoria,
        'resultado_real':{'r_pct':float(100*teste.real_r.sum()/(teste.real_r+teste.real_d).sum()),
                          'd_pct':float(100*teste.real_d.sum()/(teste.real_r+teste.real_d).sum()),
                          'eleitorais_landon':int(teste.loc[teste.real_r>teste.real_d,'eleitorais'].sum()),
                          'eleitorais_roosevelt':int(teste.loc[teste.real_d>teste.real_r,'eleitorais'].sum()),'vencedor':'Roosevelt'},
        'coeficientes_linear':{k:float(v) for k,v in zip(picks['linear']['features'],lr.coef_)},
        'intercepto_linear_pp':float(lr.intercept_),
        'referencia_validacao_1932':metricas(validacao.real_rd_pct,validacao.real_anterior),
        'pesquisa_validacao_1932':metricas(validacao.real_rd_pct,validacao.pesquisa_rd_pct),
        'fonte_sha256':hashlib.sha256(Path(arquivo).read_bytes()).hexdigest()}
    teste['vencedor_real']=np.where(teste.real_r>teste.real_d,'Landon','Roosevelt')
    completa.to_csv(saida/'base_limpa_1924_1936.csv',index=False,encoding='utf-8-sig')
    pares.to_csv(saida/'treino_96_pares.csv',index=False,encoding='utf-8-sig')
    teste.to_csv(saida/'previsoes_1936.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(resultados).to_csv(saida/'comparacao_modelos_1936.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(exclusoes).to_csv(saida/'exclusoes_etl.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(auditoria).to_csv(saida/'auditoria_etl.csv',index=False,encoding='utf-8-sig')
    (saida/'resumo_resultados.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2))
    graficos(pares,teste,modelos,picks,resumo,saida)
    print(json.dumps(resumo,ensure_ascii=False,indent=2))
    return resumo

def graficos(treino,teste,modelos,picks,resumo,saida):
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':13,'figure.facecolor':'#f5f1e9','axes.facecolor':'#f5f1e9','axes.spines.top':False,'axes.spines.right':False})
    fig,axes=plt.subplots(1,2,figsize=(11.4,4.7),sharex=True,sharey=True)
    for ax,tipo,titulo,cor in zip(axes,['linear','svr'],['Regressão linear','SVM para regressão (SVR)'],['#697a92','#007f86']):
        p=prever(modelos[tipo],treino[picks[tipo]['features']])
        ax.scatter(treino.real_rd_pct,p,c=cor,s=28,alpha=.7)
        ax.plot([0,100],[0,100],'--',c='#d56549',lw=1,label='Ajuste perfeito')
        ax.set(title=titulo,xlabel='Voto real histórico · % republicano',xlim=(0,100),ylim=(0,100));ax.grid(alpha=.12)
    axes[0].set_ylabel('Percentual estimado no treino')
    axes[0].legend(fontsize=9,frameon=False)
    fig.tight_layout();fig.savefig(saida/'treino_modelos_lado_a_lado.png',dpi=240);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11.4,4.7),sharex=True,sharey=True)
    for ax,tipo,titulo,cor in zip(axes,['linear','svr'],['Regressão linear','SVM / SVR'],['#697a92','#007f86']):
        erro=abs(teste[tipo]-teste.real_rd_pct).mean()
        ax.scatter(teste.real_rd_pct,teste[tipo],c=cor,s=32)
        ax.plot([0,100],[0,100],'--',c='#d56549',lw=1,label='Previsão perfeita')
        ax.set(title=f'{titulo}\nErro médio: {erro:.2f} p.p.',xlabel='Resultado real de Landon (%)',xlim=(0,100),ylim=(0,100));ax.grid(alpha=.12);ax.legend(fontsize=9,frameon=False)
    axes[0].set_ylabel('Previsão para Landon (%)')
    fig.tight_layout();fig.savefig(saida/'avaliacao_1936_lado_a_lado.png',dpi=240);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,3.7))
    met=resumo['resultados_1936'];vals=[x['mae_pp'] for x in met]
    bars=ax.barh([x['metodo'] for x in met],vals,color=['#b7bec9','#d3a94f','#697a92','#007f86'])
    for bar,v in zip(bars,vals):ax.text(v+.3,bar.get_y()+bar.get_height()/2,f'{v:.2f}',va='center',fontsize=10)
    ax.invert_yaxis();ax.set_xlim(0,max(vals)+4);ax.set_xlabel('Erro médio estadual · pontos percentuais')
    fig.tight_layout();fig.savefig(saida/'comparacao_erros.png',dpi=240);plt.close(fig)
    # Com dois ou três atributos não há uma única reta 2D que descreva todo o modelo.
    # Mostramos cortes individuais: variar a pesquisa de 1936, fixar o histórico do estado.
    fig,axes=plt.subplots(1,2,figsize=(11.4,4.7),sharex=True,sharey=True)
    grid=np.linspace(0,100,101)
    for ax,estado in zip(axes,['CALIFORNIA','MASSACHUSETTS']):
        r=teste.loc[teste.estado==estado].iloc[0]
        entrada=pd.DataFrame({'real_anterior':r.real_anterior,'pesquisa_rd_pct':grid,'pesquisa_anterior':r.pesquisa_anterior,'variacao_pesquisa':grid-r.pesquisa_anterior})
        for tipo,label,cor in [('linear','Linear','#697a92'),('svr','SVM','#007f86')]:
            ax.plot(grid,prever(modelos[tipo],entrada[picks[tipo]['features']]),c=cor,lw=2,label=label)
            ax.scatter([r.pesquisa_rd_pct],[r[tipo]],c=cor,s=65)
        ax.axhline(r.real_rd_pct,c='#d56549',ls='--',label='Resultado real 1936')
        ax.set(title=estado.title(),xlabel='Pesquisa de 1936 · % Landon',xlim=(0,100),ylim=(0,100));ax.grid(alpha=.12)
    axes[0].set_ylabel('Previsão para Landon (%)')
    handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc='lower center',ncol=3,fontsize=10,frameon=False)
    fig.tight_layout(rect=(0,.10,1,1));fig.savefig(saida/'regressoes_individuais_estados.png',dpi=240);plt.close(fig)

from google.colab import files

enviados = files.upload()
arquivo = next(
    nome for nome in enviados
    if nome.lower().endswith(".xlsx")
)

executar(arquivo, "resultados")

Saving LitDigestFull.xlsx to LitDigestFull.xlsx
{
  "protocolo": {
    "anos_usados": [
      1924,
      1928,
      1932
    ],
    "n_estados_por_ano": 48,
    "treino_validacao": "1924 fornece atributos anteriores de 1928; ajustar em 1928 e avaliar em 1932 usando 1928 como referência.",
    "treino_final": "96 pares estado-eleição: 1924→1928 e 1928→1932.",
    "previsao": "1932→1936, usando o resultado de 1932 e pesquisas de 1932/1936.",
    "selecao": "Menor MAE em 1932; empate: menos atributos e menor C.",
    "linear": {
      "esquema": "historico_e_variacao",
      "features": [
        "real_anterior",
        "variacao_pesquisa"
      ],
      "kernel": null,
      "C": null,
      "epsilon_pp": null,
      "mae_validacao_1932": 3.3319499125303658,
      "acertos_validacao_1932": 44
    },
    "svr": {
      "esquema": "historico_e_duas_pesquisas",
      "features": [
        "real_anterior",
        "pesquisa_rd_pct",
        "pesquisa_anterior"
      ],
      "kernel": "li

{'protocolo': {'anos_usados': [1924, 1928, 1932],
  'n_estados_por_ano': 48,
  'treino_validacao': '1924 fornece atributos anteriores de 1928; ajustar em 1928 e avaliar em 1932 usando 1928 como referência.',
  'treino_final': '96 pares estado-eleição: 1924→1928 e 1928→1932.',
  'previsao': '1932→1936, usando o resultado de 1932 e pesquisas de 1932/1936.',
  'selecao': 'Menor MAE em 1932; empate: menos atributos e menor C.',
  'linear': {'esquema': 'historico_e_variacao',
   'features': ['real_anterior', 'variacao_pesquisa'],
   'kernel': None,
   'C': None,
   'epsilon_pp': None,
   'mae_validacao_1932': 3.3319499125303658,
   'acertos_validacao_1932': 44},
  'svr': {'esquema': 'historico_e_duas_pesquisas',
   'features': ['real_anterior', 'pesquisa_rd_pct', 'pesquisa_anterior'],
   'kernel': 'linear',
   'C': 1.0,
   'epsilon_pp': 1.0,
   'mae_validacao_1932': 4.1058698251234125,
   'acertos_validacao_1932': 43},
  'limite': 'Só uma eleição de validação. Estudo retrospectivo; 1936 já 